# Salinan untuk GitHub
Output, attachments, metadata runtime, dan embedded data identitas telah dihapus. Metodologi asal dipertahankan dengan penyesuaian path lokal. Lihat `../docs/NOTEBOOKS.md` untuk perubahan dan batas reproduksi.


# Pipeline-Driven Exploratory Data Analysis
## COMPfest 18 Data Science Academy — Identity Document OCR

### Why this notebook exists

EDA di notebook ini **bukan sekadar kumpulan histogram**. Setiap analisis dirancang untuk menjawab keputusan arsitektur pada pipeline final tim:

```text
Heterogeneous capture quality
        ↓
Quality-aware preprocessing
(Raw / Gray / CLAHE)

Heterogeneous document structure
        ↓
PP-OCRv6 Medium detector
+ OpenOCR SVTRv2 Server recognizer
        ↓
Document-family routing
        ↓
Specialist field extractors

Repeated captures of the same labelled identity
        ↓
Identity-safe validation
        ↓
Siamese identity linking
        ↓
Leave-one-out peer consensus
```

### Methodological guardrail

Notebook membedakan dua jenis EDA:

- **RAW EDA** — hanya memakai image + ground truth.
- **PIPELINE-ASSISTED DIAGNOSTIC EDA** — memakai frozen output pipeline final untuk menguji *mengapa* routing, preprocessing, identity linking, dan consensus masuk akal.

Pipeline-assisted outputs **tidak diperlakukan sebagai fakta ground-truth**.

### Compute

Notebook ini **CPU-only**. Tidak ada OCR rerun, SVTRv2 inference, atau Siamese training.

# 0. Required Inputs

Attach dua sumber berikut ke Kaggle:

1. **Original COMPfest dataset**
   - `ground_truth.xlsx`
   - folder image / dataset ZIP

2. **Final teammate pipeline bundle**
   - `compfest.tar.gz`
   - atau extracted `V27_REPRODUCIBLE_BUNDLE`

Notebook mencari file berdasarkan **signature**, bukan nama folder Kaggle.

> `ground_truth.xlsx` diprioritaskan daripada CSV karena versi CSV diketahui kehilangan variasi record tertentu.

In [ ]:
# ============================================================
# 1. Imports, reproducibility, visual system
# ============================================================
import os
import re
import io
import json
import math
import tarfile
import zipfile
import shutil
import random
import warnings
import unicodedata
from pathlib import Path
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

from PIL import Image, ImageOps, ImageDraw, ImageFont
from IPython.display import display

import cv2

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------
# Natural notebook visual palette
# ------------------------------------------------------------
# Main idea:
# - use one blue family for most statistical plots
# - use orange only as an attention/accent color
# - use gray for background/reference/minor categories
# - avoid rainbow categorical palettes unless category meaning requires it

COLORS = {
    "blue": "#5B8DB8",
    "blue_light": "#A9C7DE",
    "blue_mid": "#7EA8C9",
    "blue_dark": "#3D6F98",
    "orange": "#E69F45",
    "orange_light": "#F4C78D",
    "green": "#6BAA75",
    "red": "#D66A6A",
    "gray": "#A7ADB4",
    "gray_dark": "#666D75",
    "light_gray": "#E7EAED",
    "ink": "#2B2B2B",
}

BLUE_SHADES = [
    "#BFD7E8",
    "#9FC2DB",
    "#7FADD0",
    "#6297C2",
    "#4C82B2",
    "#3D6F98",
]

plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "figure.dpi": 120,
    "axes.titlesize": 12.5,
    "axes.titleweight": "normal",
    "axes.labelsize": 10.5,
    "axes.edgecolor": "#BFC4C9",
    "axes.linewidth": 0.8,
    "axes.spines.top": True,
    "axes.spines.right": True,
    "grid.color": "#D9DDE1",
    "grid.alpha": 0.65,
    "grid.linewidth": 0.8,
    "font.size": 10,
    "text.color": COLORS["ink"],
    "axes.labelcolor": COLORS["ink"],
    "xtick.color": "#4D5258",
    "ytick.color": "#4D5258",
})

from pathlib import Path
import os
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "pipeline" / "run_v27.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Jalankan notebook dari folder repository ini.")

SEARCH_ROOTS = [REPO_ROOT / "pipeline", REPO_ROOT / "data/private"]
if os.environ.get("OCR_DATASET_DIR"):
    SEARCH_ROOTS.insert(0, Path(os.environ["OCR_DATASET_DIR"]).expanduser().resolve())


WORK_DIR = (
    Path("/kaggle/working/pipeline_driven_eda")
    if Path("/kaggle/working").exists()
    else Path("./pipeline_driven_eda")
)
WORK_DIR.mkdir(parents=True, exist_ok=True)

SLIDE_DIR = WORK_DIR / "slide_ready_assets"
SLIDE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 78)
print("PIPELINE-DRIVEN EDA")
print("=" * 78)
print("Seed       :", SEED)
print("Work dir   :", WORK_DIR)
print("Slide dir  :", SLIDE_DIR)
print("Compute    : CPU only")
print("✅ Environment ready.")

In [ ]:
# ============================================================
# 2. Plot helpers
# ============================================================
def finish_plot(
    title,
    xlabel=None,
    ylabel=None,
    grid=False,
    legend=True,
    save_name=None,
):
    """
    Minimal helper for natural notebook plots.
    Grid and annotation are only used when they genuinely help.
    """
    ax = plt.gca()

    if title:
        ax.set_title(title, pad=10)

    if xlabel:
        ax.set_xlabel(xlabel)

    if ylabel:
        ax.set_ylabel(ylabel)

    if grid:
        ax.grid(True, alpha=0.55)

    if not legend and ax.get_legend() is not None:
        ax.get_legend().remove()

    plt.tight_layout()

    if save_name:
        path = SLIDE_DIR / save_name
        plt.savefig(
            path,
            dpi=220,
            bbox_inches="tight",
            facecolor="white",
        )
        print("Saved:", path)

    plt.show()


def annotate_vertical_bars(ax, fmt="{:.0f}", offset=3):
    for p in ax.patches:
        h = p.get_height()
        if np.isnan(h):
            continue
        ax.annotate(
            fmt.format(h),
            (p.get_x() + p.get_width()/2, h),
            ha="center",
            va="bottom",
            xytext=(0, offset),
            textcoords="offset points",
            fontsize=9,
        )


def annotate_horizontal_bars(ax, fmt="{:.0f}", offset=4):
    for p in ax.patches:
        w = p.get_width()
        if np.isnan(w):
            continue
        ax.annotate(
            fmt.format(w),
            (w, p.get_y() + p.get_height()/2),
            ha="left",
            va="center",
            xytext=(offset, 0),
            textcoords="offset points",
            fontsize=9,
        )


print("✅ Minimal plot helpers ready.")


### Visual style used in this notebook

The visual design intentionally follows a **normal data-science notebook style**.

- plot type follows the analytical need;
- grids are used only when they help reading values;
- annotations are added only when they carry useful information;
- most plots use a restrained **blue family**, with orange/gray as semantic accents;
- explanatory text stays in markdown rather than inside the chart.

The goal is to keep the EDA visually clear and natural, while still avoiding flat one-color figures.

# 1. Data Discovery & Integrity

Notebook akan:

1. mencari `ground_truth.xlsx`,
2. mencari image directory,
3. mencari final pipeline bundle,
4. membaca **V28 released prediction table** dan route audit,
5. mengecek 632 labeled + 100 unlabeled.

Tidak ada model yang dijalankan ulang.

In [ ]:
# ============================================================
# 3. Discover original dataset
# ============================================================
def discover_ground_truth_folder():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        matches = list(root.rglob("ground_truth.xlsx"))
        for p in matches:
            image_candidates = [
                p.parent / "images",
                p.parent,
            ]
            n_images = max(
                (
                    len(list(folder.glob("image_*.jpg")))
                    if folder.exists() else 0
                )
                for folder in image_candidates
            )
            if n_images >= 600:
                return {
                    "type": "folder",
                    "gt_path": p,
                    "base_dir": p.parent,
                }
    return None


def discover_dataset_zip():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        for zp in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zp, "r") as z:
                    names = z.namelist()
                    gt_matches = [
                        n for n in names
                        if n.endswith("ground_truth.xlsx")
                    ]
                    img_matches = [
                        n for n in names
                        if re.search(r"/images/image_\d+\.(jpg|jpeg|png)$", n, re.I)
                    ]

                    if gt_matches and len(img_matches) >= 600:
                        return {
                            "type": "zip",
                            "zip_path": zp,
                            "gt_member": gt_matches[0],
                            "image_members": img_matches,
                        }
            except zipfile.BadZipFile:
                continue
    return None


DATASET_SOURCE = discover_ground_truth_folder()

if DATASET_SOURCE is None:
    DATASET_SOURCE = discover_dataset_zip()

if DATASET_SOURCE is None:
    raise FileNotFoundError(
        "Original COMPfest dataset tidak ditemukan. "
        "Attach folder dataset atau ZIP yang berisi ground_truth.xlsx + images."
    )

print("=" * 78)
print("ORIGINAL DATASET")
print("=" * 78)
for k, v in DATASET_SOURCE.items():
    if k != "image_members":
        print(f"{k:14s}: {v}")

if DATASET_SOURCE["type"] == "zip":
    print("images        :", len(DATASET_SOURCE["image_members"]))

In [ ]:
# ============================================================
# 4. Resolve GT + image paths
# ============================================================
if DATASET_SOURCE["type"] == "folder":
    GT_PATH = DATASET_SOURCE["gt_path"]
    BASE_DIR = DATASET_SOURCE["base_dir"]

    candidate_image_dirs = [
        BASE_DIR / "images",
        BASE_DIR,
    ]

    IMAGE_DIR = max(
        candidate_image_dirs,
        key=lambda p: len(list(p.glob("image_*.jpg")))
        if p.exists() else 0,
    )

else:
    EXTRACT_DIR = WORK_DIR / "original_dataset"
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(
        DATASET_SOURCE["zip_path"],
        "r",
    ) as z:
        gt_member = DATASET_SOURCE["gt_member"]
        extracted_gt = z.extract(gt_member, EXTRACT_DIR)

        for member in DATASET_SOURCE["image_members"]:
            target_name = Path(member).name
            target_path = EXTRACT_DIR / "images" / target_name

            if not target_path.exists():
                target_path.parent.mkdir(
                    parents=True,
                    exist_ok=True,
                )
                with z.open(member) as src, open(
                    target_path,
                    "wb",
                ) as dst:
                    shutil.copyfileobj(src, dst)

    GT_PATH = Path(extracted_gt)
    IMAGE_DIR = EXTRACT_DIR / "images"

print("GT path   :", GT_PATH)
print("Image dir :", IMAGE_DIR)
print("Images    :", len(list(IMAGE_DIR.glob("image_*.*"))))
print("✅ Original dataset ready.")

In [ ]:
# ============================================================
# 5. Discover final teammate pipeline bundle
# ============================================================
V28_PRED_SUFFIX = (
    "results_v28_generic_unknown_peer_consensus/"
    "predictions_released.csv"
)
GENERIC_AUDIT_SUFFIX = (
    "reports/GENERIC_ID_NAME_AUDIT.csv"
)
FINAL_REPORT_SUFFIX = (
    "reports/FINAL_V27_REPORT.md"
)

def discover_pipeline_tar():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        candidates = (
            list(root.rglob("*.tar.gz"))
            + list(root.rglob("*.tgz"))
        )

        for tp in candidates:
            try:
                with tarfile.open(tp, "r:*") as t:
                    names = t.getnames()

                    if any(
                        n.endswith(V28_PRED_SUFFIX)
                        for n in names
                    ):
                        return {
                            "type": "tar",
                            "path": tp,
                        }
            except tarfile.TarError:
                continue

    return None


def discover_pipeline_folder():
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue

        matches = list(
            root.rglob("predictions_released.csv")
        )

        for p in matches:
            if (
                "results_v28_generic_unknown_peer_consensus"
                in str(p)
            ):
                return {
                    "type": "folder",
                    "prediction_path": p,
                }

    return None


PIPELINE_SOURCE = discover_pipeline_tar()

if PIPELINE_SOURCE is None:
    PIPELINE_SOURCE = discover_pipeline_folder()

if PIPELINE_SOURCE is None:
    raise FileNotFoundError(
        "Final teammate pipeline bundle tidak ditemukan. "
        "Attach compfest.tar.gz atau extracted V27_REPRODUCIBLE_BUNDLE."
    )

print("=" * 78)
print("TEAMMATE FINAL PIPELINE ARTIFACT")
print("=" * 78)
print(PIPELINE_SOURCE)

In [ ]:
# ============================================================
# 6. Load raw GT + V28 pipeline artifact
# ============================================================
gt = pd.read_excel(GT_PATH)

if PIPELINE_SOURCE["type"] == "tar":
    with tarfile.open(
        PIPELINE_SOURCE["path"],
        "r:*",
    ) as t:
        names = t.getnames()

        def member_ending(suffix):
            found = [
                n for n in names
                if n.endswith(suffix)
            ]
            if not found:
                return None
            return found[0]

        pred_member = member_ending(V28_PRED_SUFFIX)
        audit_member = member_ending(GENERIC_AUDIT_SUFFIX)
        report_member = member_ending(FINAL_REPORT_SUFFIX)

        pipeline = pd.read_csv(
            t.extractfile(pred_member)
        )

        generic_audit = (
            pd.read_csv(t.extractfile(audit_member))
            if audit_member else pd.DataFrame()
        )

        final_report_text = (
            t.extractfile(report_member)
            .read()
            .decode("utf-8", errors="ignore")
            if report_member else ""
        )

else:
    pred_path = PIPELINE_SOURCE["prediction_path"]
    bundle_root = pred_path.parent.parent

    pipeline = pd.read_csv(pred_path)

    audit_path = (
        bundle_root / "reports"
        / "GENERIC_ID_NAME_AUDIT.csv"
    )
    generic_audit = (
        pd.read_csv(audit_path)
        if audit_path.exists()
        else pd.DataFrame()
    )

    report_path = (
        bundle_root / "reports"
        / "FINAL_V27_REPORT.md"
    )
    final_report_text = (
        report_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )
        if report_path.exists()
        else ""
    )

labeled_pipe = pipeline[
    pipeline["has_ground_truth"].astype(bool)
].copy()

unlabeled_pipe = pipeline[
    ~pipeline["has_ground_truth"].astype(bool)
].copy()

assert len(gt) == 632
assert len(labeled_pipe) == 632
assert len(unlabeled_pipe) == 100
assert set(gt["filename"]) == set(labeled_pipe["filename"])

print("=" * 78)
print("INTEGRITY CHECK")
print("=" * 78)
print("Raw GT rows        :", len(gt))
print("Pipeline rows      :", len(pipeline))
print("Labeled artifact   :", len(labeled_pipe))
print("Unlabeled artifact :", len(unlabeled_pipe))
print("✅ 632 labeled + 100 unlabeled confirmed.")

# 2. Identity Audit — **132 or 152?**

Ini sengaja diletakkan sebelum EDA utama karena angka identity harus konsisten dengan pipeline final.

Kita akan membandingkan:

1. unique normalized **Name**,
2. unique normalized **(Name, DOB)**,
3. final pipeline `identity_group`,
4. nama yang menyebabkan perbedaan count.

### Terminology used in this notebook

- **Unique normalized names** ≠ selalu unique people.
- **Pipeline identity groups** = label-derived grouping yang digunakan pipeline final untuk training / validation identity logic.
- Karena itu, EDA utama akan menggunakan **pipeline identity groups** ketika membahas repeated captures.

In [ ]:
# ============================================================
# 7. Identity count audit
# ============================================================
def norm_text(v):
    if pd.isna(v):
        return ""
    return re.sub(
        r"\s+",
        " ",
        unicodedata.normalize(
            "NFKC",
            str(v),
        ).upper().strip(),
    )


def norm_date_label(v):
    if pd.isna(v):
        return ""

    parsed = pd.to_datetime(
        v,
        errors="coerce",
    )

    if pd.notna(parsed):
        return parsed.strftime("%Y-%m-%d")

    return norm_text(v)


identity_audit = gt.copy()
identity_audit["name_norm"] = (
    identity_audit["name"].map(norm_text)
)
identity_audit["dob_norm"] = (
    identity_audit["birth_date"].map(
        norm_date_label
    )
)

n_unique_names = (
    identity_audit["name_norm"].nunique()
)

n_name_dob = (
    identity_audit[
        ["name_norm", "dob_norm"]
    ]
    .drop_duplicates()
    .shape[0]
)

n_pipeline_groups = (
    labeled_pipe["identity_group"].nunique()
)

identity_count_table = pd.DataFrame({
    "Definition": [
        "Unique normalized Name",
        "Unique normalized (Name, DOB)",
        "Final pipeline identity_group",
    ],
    "Count": [
        n_unique_names,
        n_name_dob,
        n_pipeline_groups,
    ],
    "Interpretation": [
        "String-level uniqueness; may merge different people with same name",
        "Separates same name when DOB differs",
        "Grouping actually used by final teammate pipeline",
    ],
})

display(
    identity_count_table.style
    .hide(axis="index")
)

assert n_unique_names == 132
assert n_name_dob == 152
assert n_pipeline_groups == 152

print()
print("✅ VERIFIED:")
print("   132 = unique normalized Name values")
print("   152 = unique (Name, DOB) pairs")
print("   152 = final pipeline identity groups")

In [ ]:
# ============================================================
# 8. Explain exactly where 132 → 152 comes from
# ============================================================
dob_variants = (
    identity_audit
    .groupby("name_norm")["dob_norm"]
    .nunique()
    .sort_values(ascending=False)
)

names_with_multiple_dob = (
    dob_variants[dob_variants > 1]
    .reset_index()
    .rename(
        columns={
            "dob_norm": "Unique DOB labels"
        }
    )
)

rows = []

for name in names_with_multiple_dob["name_norm"]:
    sub = identity_audit[
        identity_audit["name_norm"] == name
    ]

    rows.append({
        "Name": name,
        "Images": len(sub),
        "Unique DOB labels": sub["dob_norm"].nunique(),
        "DOB examples": ", ".join(
            sorted(sub["dob_norm"].unique())[:5]
        )
        + (
            " ..."
            if sub["dob_norm"].nunique() > 5
            else ""
        ),
    })

identity_difference_explainer = pd.DataFrame(rows)

display(
    identity_difference_explainer
    .style
    .hide(axis="index")
)

print(
    "\nDifference 152 - 132 =",
    n_pipeline_groups - n_unique_names,
    "additional label-derived groups."
)

### Identity count conclusion

**Untuk notebook yang harus nyambung dengan pipeline final, kita memakai `152 pipeline identity groups` sebagai angka utama.**

Tetapi kita **tidak menyebutnya 152 verified unique humans**.

Audit memperlihatkan bahwa selisih dari 132 unique names terutama berasal dari:

- `JUSTYNA AGNIESZKA KRAWCZYK-NOWAK` — 20 scans memiliki 20 DOB label berbeda secara berurutan; pola ini sangat menyerupai annotation anomaly.
- `VZOR SPECIMEN` — satu nama specimen memiliki dua DOB berbeda dan memang dipisah menjadi dua identity labels.

Karena final teammate pipeline sendiri memakai **152 `identity_group`**, angka **152** adalah angka yang konsisten untuk identity-safe EDA dan validation story.

> **Slide-safe wording:** “632 labeled images map to 152 label-derived identity groups, with repeated captures concentrated in a small subset of groups.”

# 3. Ground-Truth Quality & Field Availability

Sebelum menggunakan field presence sebagai insight, kita audit GT.

Khusus Address:
- raw XLSX memiliki 113 non-empty address cells,
- final pipeline artifact memakai 112 operational address labels,
- satu raw entry terdeteksi menyerupai DOB yang tersalin ke Address.

Ini penting supaya EDA tidak salah menganggap annotation anomaly sebagai alamat valid.

In [ ]:
# ============================================================
# 9. Ground-truth anomaly audit
# ============================================================
gt_audit = gt.copy()

gt_audit["birth_date_parsed"] = pd.to_datetime(
    gt_audit["birth_date"],
    errors="coerce",
)

gt_audit["address_as_date"] = pd.to_datetime(
    gt_audit["address"],
    errors="coerce",
)

gt_audit["address_equals_dob"] = (
    gt_audit["address_as_date"].notna()
    & gt_audit["birth_date_parsed"].notna()
    & (
        gt_audit["address_as_date"].dt.date
        == gt_audit["birth_date_parsed"].dt.date
    )
)

suspicious_gt = gt_audit[
    gt_audit["address_equals_dob"]
][
    [
        "filename",
        "name",
        "birth_date",
        "address",
    ]
]

print("Raw non-empty Address :", gt["address"].notna().sum())
print(
    "Pipeline operational Address-present :",
    int(labeled_pipe["address_present_gt"].sum()),
)
print(
    "Suspicious Address copied from DOB    :",
    int(gt_audit["address_equals_dob"].sum()),
)

display(
    suspicious_gt.style.hide(axis="index")
)

In [ ]:
# ============================================================
# 10. Dataset KPI table
# ============================================================
captures = (
    labeled_pipe
    .groupby("identity_group")
    .size()
)

overview = pd.DataFrame({
    "Metric": [
        "Total images",
        "Labeled images",
        "Unlabeled images",
        "Pipeline identity groups",
        "Repeated identity groups",
        "Singleton identity groups",
        "Largest capture group",
        "Operational Address-present",
        "Operational Address-empty",
    ],
    "Value": [
        len(pipeline),
        len(labeled_pipe),
        len(unlabeled_pipe),
        labeled_pipe["identity_group"].nunique(),
        int((captures > 1).sum()),
        int((captures == 1).sum()),
        int(captures.max()),
        int(labeled_pipe["address_present_gt"].sum()),
        int((~labeled_pipe["address_present_gt"]).sum()),
    ],
})

display(
    overview.style
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 11. Field availability
# ============================================================
field_availability = pd.DataFrame({
    "Field": ["Name", "Birth Date", "Address"],
    "Present": [
        len(labeled_pipe),
        len(labeled_pipe),
        int(labeled_pipe["address_present_gt"].sum()),
    ],
})

field_availability["Missing"] = (
    len(labeled_pipe) - field_availability["Present"]
)
field_availability["Present rate"] = (
    field_availability["Present"] / len(labeled_pipe)
)

x = np.arange(len(field_availability))
colors = [
    COLORS["blue_mid"],
    COLORS["blue"],
    COLORS["orange"],
]

plt.figure(figsize=(8.5, 5.2))
plt.bar(
    x,
    field_availability["Present"],
    color=colors,
    width=0.62,
)

plt.xticks(x, field_availability["Field"])
plt.ylim(0, 690)

# Only annotate the values that matter here
for i, row in field_availability.iterrows():
    plt.text(
        i,
        row["Present"] + 9,
        f'{row["Present"]} ({row["Present rate"]:.1%})',
        ha="center",
        fontsize=9,
    )

finish_plot(
    "Field Availability",
    ylabel="Labeled documents",
    grid=True,
    legend=False,
    save_name="slide_01_field_availability.png",
)

### Key finding → pipeline implication

**KEY FINDING**  
Address is not merely difficult — it is **structurally absent** from most documents.

**MODELING IMPLICATION**  
A universal “always extract Address” rule is inappropriate.

**PIPELINE RESPONSE**  
Document-family routing and field-specific parsing are justified because expected field availability changes by document family.

# 4. Identity Repetition & Leakage Risk

Sekarang kita pakai **152 final pipeline identity groups**.

Pertanyaan:
- berapa group yang repeated?
- berapa banyak image sebenarnya berasal dari repeated identities?
- seberapa berbahaya random image split?

In [ ]:
# ============================================================
# 12. Capture distribution per pipeline identity group
# ============================================================
capture_dist = (
    captures
    .value_counts()
    .sort_index()
    .rename_axis("Captures per identity group")
    .reset_index(name="Identity groups")
)

capture_dist["Share of identity groups"] = (
    capture_dist["Identity groups"]
    / capture_dist["Identity groups"].sum()
)

display(
    capture_dist.style
    .format({
        "Share of identity groups": "{:.1%}"
    })
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 13. Repeated capture structure
# ============================================================
plot_capture = capture_dist.copy()

capture_colors = [
    COLORS["gray"] if n == 1
    else COLORS["blue_mid"] if n <= 2
    else COLORS["blue"]
    for n in plot_capture["Captures per identity group"]
]

plt.figure(figsize=(9, 5.2))
plt.bar(
    plot_capture["Captures per identity group"].astype(str),
    plot_capture["Identity groups"],
    color=capture_colors,
    width=0.65,
)

finish_plot(
    "Distribution of Captures per Identity Group",
    xlabel="Captures per identity group",
    ylabel="Number of identity groups",
    grid=True,
    legend=False,
    save_name="slide_02_identity_repetition.png",
)

In [ ]:
# ============================================================
# 14. Repeated-capture concentration
# ============================================================
repeated_group_count = int(
    (captures > 1).sum()
)
repeated_rows = int(
    captures[captures > 1].sum()
)
singleton_rows = int(
    captures[captures == 1].sum()
)

identity_concentration = pd.DataFrame({
    "Segment": [
        "Repeated identity groups",
        "Singleton identity groups",
        "Images from repeated groups",
        "Images from singleton groups",
    ],
    "Count": [
        repeated_group_count,
        int((captures == 1).sum()),
        repeated_rows,
        singleton_rows,
    ],
})

display(
    identity_concentration
    .style.hide(axis="index")
)

print(
    f"\n{repeated_rows / len(labeled_pipe):.1%} "
    "of labeled images belong to repeated identity groups."
)

In [ ]:
# ============================================================
# 15. Random image split leakage simulation
# ============================================================
def leakage_rate_for_split(
    frame,
    val_frac=0.20,
    seed=0,
):
    rs = np.random.default_rng(seed)

    n_val = int(
        round(len(frame) * val_frac)
    )

    val_idx = rs.choice(
        frame.index.to_numpy(),
        size=n_val,
        replace=False,
    )

    val_groups = set(
        frame.loc[
            val_idx,
            "identity_group",
        ]
    )

    train_groups = set(
        frame.drop(index=val_idx)[
            "identity_group"
        ]
    )

    repeated_groups = set(
        captures[captures > 1].index
    )

    leaked = (
        val_groups
        & train_groups
        & repeated_groups
    )

    return (
        len(leaked)
        / max(len(repeated_groups), 1)
    )


leakage_trials = np.array([
    leakage_rate_for_split(
        labeled_pipe,
        seed=s,
    )
    for s in range(500)
])

leakage_summary = pd.DataFrame({
    "Statistic": [
        "Mean repeated-group leakage",
        "5th percentile",
        "95th percentile",
    ],
    "Value": [
        leakage_trials.mean(),
        np.quantile(
            leakage_trials,
            0.05,
        ),
        np.quantile(
            leakage_trials,
            0.95,
        ),
    ],
})

display(
    leakage_summary.style
    .format({"Value": "{:.1%}"})
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 16. Leakage distribution plot
# ============================================================
plt.figure(figsize=(8.8, 5.2))

counts, bins, patches = plt.hist(
    100 * leakage_trials,
    bins=np.arange(60, 102, 2),
    edgecolor="white",
)

# Subtle gradient inside one color family
for i, p in enumerate(patches):
    shade = BLUE_SHADES[min(
        int(i / max(len(patches), 1) * len(BLUE_SHADES)),
        len(BLUE_SHADES) - 1
    )]
    p.set_facecolor(shade)

plt.axvline(
    100 * leakage_trials.mean(),
    color=COLORS["orange"],
    linewidth=2,
    linestyle="--",
    label=f"Mean = {100*leakage_trials.mean():.1f}%",
)

plt.legend(frameon=False)

finish_plot(
    "Leakage Risk from Random Image Splits",
    xlabel="Repeated identity groups leaked (%)",
    ylabel="Simulation count",
    grid=True,
    legend=True,
)

### Key finding → pipeline implication

**KEY FINDING**  
A small set of repeated identity groups contributes a very large share of labeled images.

**MODELING IMPLICATION**  
Random image-level validation can place different captures of the same label-derived identity in train and validation.

**PIPELINE RESPONSE**
1. identity-aware validation is required;
2. repeated scans are also an inference opportunity for identity linking and peer consensus.

# 5. RAW Image Quality Heterogeneity

Pipeline final uses quality-aware preprocessing, so EDA harus menunjukkan bahwa degradation memang tidak homogen.

Di bagian ini kita menghitung ulang fitur ringan **langsung dari original image**, tanpa memakai output OCR:

- brightness,
- contrast,
- blur score,
- glare ratio,
- illumination variation,
- aspect ratio.

Image di-resize maksimal 900 px untuk mempercepat EDA tanpa mengubah karakter global kualitas.

In [ ]:
# ============================================================
# 17. Extract lightweight raw image-quality features
# ============================================================
QUALITY_CACHE = (
    WORK_DIR / "raw_image_quality_eda.csv"
)

def image_quality_features(path):
    img = cv2.imread(str(path))

    if img is None:
        raise ValueError(
            f"Unable to read image: {path}"
        )

    h0, w0 = img.shape[:2]

    scale = min(
        1.0,
        900.0 / max(h0, w0),
    )

    if scale < 1:
        img = cv2.resize(
            img,
            (
                int(w0 * scale),
                int(h0 * scale),
            ),
            interpolation=cv2.INTER_AREA,
        )

    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY,
    )

    brightness = float(
        gray.mean()
    )
    contrast = float(
        gray.std()
    )
    blur_score = float(
        cv2.Laplacian(
            gray,
            cv2.CV_64F,
        ).var()
    )

    glare_ratio = float(
        (gray >= 245).mean()
    )

    illumination_map = cv2.GaussianBlur(
        gray,
        (0, 0),
        sigmaX=25,
        sigmaY=25,
    )

    illumination_variation = float(
        illumination_map.std()
    )

    return {
        "width": w0,
        "height": h0,
        "aspect_ratio_raw": w0 / h0,
        "brightness_raw": brightness,
        "contrast_raw": contrast,
        "blur_score_raw": blur_score,
        "glare_ratio_raw": glare_ratio,
        "illumination_variation_raw": (
            illumination_variation
        ),
    }


if QUALITY_CACHE.exists():
    raw_quality = pd.read_csv(
        QUALITY_CACHE
    )
    print(
        "Loaded cached quality features:",
        QUALITY_CACHE,
    )

else:
    quality_rows = []
    image_files = sorted(
        IMAGE_DIR.glob("image_*.*")
    )

    for i, path in enumerate(
        image_files,
        1,
    ):
        feats = image_quality_features(
            path
        )
        feats["filename"] = path.name
        quality_rows.append(feats)

        if i % 100 == 0:
            print(
                f"Processed {i}/{len(image_files)} images"
            )

    raw_quality = pd.DataFrame(
        quality_rows
    )

    raw_quality.to_csv(
        QUALITY_CACHE,
        index=False,
    )

assert len(raw_quality) >= 732

print(
    "✅ Raw quality features:",
    raw_quality.shape,
)

In [ ]:
# ============================================================
# 18. Merge raw quality with pipeline diagnostics
# ============================================================
eda = (
    labeled_pipe
    .merge(
        raw_quality,
        on="filename",
        how="left",
        validate="one_to_one",
    )
)

assert len(eda) == 632
assert eda[
    "brightness_raw"
].notna().all()

quality_stats = (
    eda[
        [
            "brightness_raw",
            "contrast_raw",
            "blur_score_raw",
            "glare_ratio_raw",
            "illumination_variation_raw",
            "aspect_ratio_raw",
        ]
    ]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
        ]
    )
    .T
)

display(
    quality_stats[
        [
            "min",
            "5%",
            "25%",
            "50%",
            "75%",
            "95%",
            "max",
        ]
    ]
    .round(3)
)

In [ ]:
# ============================================================
# 19. Brightness distribution
# ============================================================
plt.figure(figsize=(8.8, 5.2))

counts, bins, patches = plt.hist(
    eda["brightness_raw"],
    bins=30,
    edgecolor="white",
)

for i, p in enumerate(patches):
    shade = BLUE_SHADES[min(
        int(i / max(len(patches), 1) * len(BLUE_SHADES)),
        len(BLUE_SHADES) - 1
    )]
    p.set_facecolor(shade)

median_brightness = eda["brightness_raw"].median()

plt.axvline(
    median_brightness,
    color=COLORS["orange"],
    linewidth=2,
    linestyle="--",
    label=f"Median = {median_brightness:.1f}",
)

plt.legend(frameon=False)

finish_plot(
    "Distribution of Image Brightness",
    xlabel="Mean grayscale brightness (0–255)",
    ylabel="Frequency",
    grid=True,
    legend=True,
)

In [ ]:
# ============================================================
# 20. Sharpness distribution
# ============================================================
plot_blur = eda["blur_score_raw"].clip(lower=1e-3)

plt.figure(figsize=(8.8, 5.2))

counts, bins, patches = plt.hist(
    np.log10(plot_blur),
    bins=30,
    edgecolor="white",
)

for i, p in enumerate(patches):
    shade = BLUE_SHADES[min(
        int(i / max(len(patches), 1) * len(BLUE_SHADES)),
        len(BLUE_SHADES) - 1
    )]
    p.set_facecolor(shade)

median_blur = np.median(np.log10(plot_blur))

plt.axvline(
    median_blur,
    color=COLORS["orange"],
    linewidth=2,
    linestyle="--",
    label=f"Median = {median_blur:.2f}",
)

plt.legend(frameon=False)

finish_plot(
    "Distribution of Image Sharpness",
    xlabel="log10(Variance of Laplacian)",
    ylabel="Frequency",
    grid=True,
    legend=True,
)

## Pipeline quality groups

Berikutnya kita **beralih ke pipeline-assisted diagnostic EDA**.

Final pipeline sudah memiliki quality flags dan `quality_group`. Kita tidak menganggap label ini ground truth kualitas; kita menggunakannya untuk melihat bagaimana preprocessing policy merespons variasi yang baru saja terlihat pada raw images.

In [ ]:
# ============================================================
# 21. Pipeline quality-group distribution
# ============================================================
quality_order = [
    "clean",
    "low_contrast",
    "dark",
    "blur",
    "severe_blur",
    "glare",
    "uneven_light",
]

quality_counts = (
    eda["quality_group"]
    .value_counts()
    .reindex(quality_order, fill_value=0)
)

quality_colors = [
    COLORS["blue"],
    COLORS["blue_mid"],
    COLORS["blue_dark"],
    COLORS["blue_light"],
    COLORS["orange"],
    COLORS["orange_light"],
    COLORS["gray"],
]

plt.figure(figsize=(9.2, 5.2))
plt.bar(
    quality_counts.index,
    quality_counts.values,
    color=quality_colors,
    width=0.68,
)

plt.xticks(rotation=20, ha="right")

finish_plot(
    "Distribution of Image Quality Groups",
    ylabel="Labeled documents",
    grid=True,
    legend=False,
    save_name="slide_03_quality_groups.png",
)

In [ ]:
# ============================================================
# 22. Quality group × selected preprocessing
# ============================================================
preprocess_order = ["raw", "gray", "gray_clahe"]

quality_preprocess = pd.crosstab(
    eda["quality_group"],
    eda["selected_preprocessing_method"],
    normalize="index",
).reindex(
    index=quality_order,
    columns=preprocess_order,
    fill_value=0,
)

plt.figure(figsize=(7.8, 5.6))
im = plt.imshow(
    quality_preprocess.values,
    cmap="Blues",
    aspect="auto",
    vmin=0,
    vmax=1,
)

plt.colorbar(
    im,
    fraction=0.035,
    pad=0.03,
    label="Share within quality group",
)

plt.xticks(
    np.arange(len(preprocess_order)),
    ["Raw", "Gray", "Gray + CLAHE"],
)
plt.yticks(
    np.arange(len(quality_order)),
    [q.replace("_", " ").title() for q in quality_order],
)

# Values are useful in a heatmap, so keep them
for i in range(quality_preprocess.shape[0]):
    for j in range(quality_preprocess.shape[1]):
        value = quality_preprocess.iloc[i, j]
        plt.text(
            j,
            i,
            f"{value:.0%}",
            ha="center",
            va="center",
            fontsize=9,
            color="white" if value > 0.55 else COLORS["ink"],
        )

plt.title("Selected Preprocessing by Quality Group")
plt.xlabel("Preprocessing method")
plt.ylabel("Quality group")
plt.tight_layout()

heatmap_path = SLIDE_DIR / "slide_04_quality_preprocessing_heatmap.png"
plt.savefig(
    heatmap_path,
    dpi=220,
    bbox_inches="tight",
    facecolor="white",
)
print("Saved:", heatmap_path)
plt.show()

### Key finding → pipeline implication

**KEY FINDING**  
Raw images cover multiple brightness, sharpness, glare, and illumination regimes. The frozen pipeline also classifies substantial non-clean subsets.

**MODELING IMPLICATION**  
Aggressive preprocessing applied universally risks altering images that are already clean, while leaving degraded captures untreated.

**PIPELINE RESPONSE**  
Quality-aware selection among **Raw / Gray / Gray+CLAHE** is a data-driven response to heterogeneous capture conditions.

# 6. Professional Image Galleries

Statistik harus didampingi visual evidence.

Gallery berikut memilih image berdasarkan **raw EDA metrics**, bukan filename block:
- darkest,
- median brightness,
- brightest,
- blurriest,
- median sharpness,
- sharpest.

In [ ]:
# ============================================================
# 23. Image montage utility (single rendered image, no chart subplot)
# ============================================================
def load_rgb(filename):
    path = IMAGE_DIR / filename
    return Image.open(path).convert("RGB")


def fit_card_image(
    img,
    size=(360, 240),
):
    canvas = Image.new(
        "RGB",
        size,
        "white",
    )

    fitted = ImageOps.contain(
        img,
        (
            size[0] - 16,
            size[1] - 16,
        ),
        Image.Resampling.LANCZOS,
    )

    x = (
        size[0] - fitted.width
    ) // 2
    y = (
        size[1] - fitted.height
    ) // 2

    canvas.paste(
        fitted,
        (x, y),
    )
    return canvas


def make_montage(
    items,
    columns=3,
    card_size=(380, 290),
    title_height=58,
    gap=14,
):
    rows = math.ceil(
        len(items) / columns
    )

    width = (
        columns * card_size[0]
        + (columns + 1) * gap
    )

    height = (
        rows
        * (
            card_size[1]
            + title_height
        )
        + (rows + 1) * gap
    )

    canvas = Image.new(
        "RGB",
        (width, height),
        "#F7F9FB",
    )

    draw = ImageDraw.Draw(
        canvas
    )

    for idx, item in enumerate(items):
        r = idx // columns
        c = idx % columns

        x = (
            gap
            + c
            * (
                card_size[0]
                + gap
            )
        )

        y = (
            gap
            + r
            * (
                card_size[1]
                + title_height
                + gap
            )
        )

        image = fit_card_image(
            load_rgb(
                item["filename"]
            ),
            card_size,
        )

        canvas.paste(
            image,
            (x, y),
        )

        draw.text(
            (x + 4, y + card_size[1] + 8),
            item["label"],
            fill=COLORS["ink"],
        )

        if item.get("detail"):
            draw.text(
                (
                    x + 4,
                    y + card_size[1] + 29,
                ),
                item["detail"],
                fill="#66727E",
            )

    return canvas


print("✅ Montage utility ready.")

In [ ]:
# ============================================================
# 24. Quality extremes montage
# ============================================================
ordered_brightness = eda.sort_values(
    "brightness_raw"
)

ordered_blur = eda.sort_values(
    "blur_score_raw"
)

median_idx_b = len(
    ordered_brightness
) // 2

median_idx_blur = len(
    ordered_blur
) // 2

quality_gallery_items = [
    {
        "filename": ordered_brightness.iloc[0]["filename"],
        "label": "Darkest capture",
        "detail": (
            f'Brightness = '
            f'{ordered_brightness.iloc[0]["brightness_raw"]:.1f}'
        ),
    },
    {
        "filename": ordered_brightness.iloc[median_idx_b]["filename"],
        "label": "Median brightness",
        "detail": (
            f'Brightness = '
            f'{ordered_brightness.iloc[median_idx_b]["brightness_raw"]:.1f}'
        ),
    },
    {
        "filename": ordered_brightness.iloc[-1]["filename"],
        "label": "Brightest capture",
        "detail": (
            f'Brightness = '
            f'{ordered_brightness.iloc[-1]["brightness_raw"]:.1f}'
        ),
    },
    {
        "filename": ordered_blur.iloc[0]["filename"],
        "label": "Lowest sharpness",
        "detail": (
            f'Blur score = '
            f'{ordered_blur.iloc[0]["blur_score_raw"]:.1f}'
        ),
    },
    {
        "filename": ordered_blur.iloc[median_idx_blur]["filename"],
        "label": "Median sharpness",
        "detail": (
            f'Blur score = '
            f'{ordered_blur.iloc[median_idx_blur]["blur_score_raw"]:.1f}'
        ),
    },
    {
        "filename": ordered_blur.iloc[-1]["filename"],
        "label": "Highest sharpness",
        "detail": (
            f'Blur score = '
            f'{ordered_blur.iloc[-1]["blur_score_raw"]:.1f}'
        ),
    },
]

quality_montage = make_montage(
    quality_gallery_items,
    columns=3,
)

quality_montage_path = (
    SLIDE_DIR
    / "slide_05_quality_extremes_montage.jpg"
)
quality_montage.save(
    quality_montage_path,
    quality=92,
)

display(quality_montage)
print("Saved:", quality_montage_path)

# 7. Document & Layout Heterogeneity

Final pipeline tidak memakai satu parser universal.

Kita gunakan:
- raw document aspect ratio,
- OCR item count,
- detailed route labels dari frozen pipeline,
- representative document images.

### Route construction

- `MYKAD` dan `MRZ_DOCUMENT` berasal dari `document_family`;
- dokumen `GENERIC_ID` dipetakan ke detailed route dari route audit:
  `EU_DRIVING_LICENSE`, `UKRAINE_PASSPORT`, `CZECH_ID`, `CHILE_ID`, dst.;
- sisanya `UNKNOWN_ROUTE`.

Ini adalah **pipeline-assisted diagnostic EDA**, bukan raw label.

In [ ]:
# ============================================================
# 25. Build detailed final-route diagnostic label
# ============================================================
route_audit = (
    generic_audit[
        [
            "filename",
            "route_status",
        ]
    ].copy()
    if len(generic_audit)
    else pd.DataFrame(
        columns=[
            "filename",
            "route_status",
        ]
    )
)

eda = eda.merge(
    route_audit,
    on="filename",
    how="left",
)

eda["route_final"] = np.select(
    [
        eda["document_family"].eq(
            "MYKAD"
        ),
        eda["document_family"].eq(
            "MRZ_DOCUMENT"
        ),
    ],
    [
        "MYKAD",
        "MRZ / PASSPORT",
    ],
    default=(
        eda["route_status"]
        .fillna("UNKNOWN_ROUTE")
    ),
)

route_counts = (
    eda["route_final"]
    .value_counts()
    .rename_axis("Route")
    .reset_index(name="Documents")
)

display(
    route_counts.style
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 26. Document-route distribution
# ============================================================
route_plot = route_counts.sort_values(
    "Documents",
    ascending=True,
)

route_colors = []
for route in route_plot["Route"]:
    if route == "UNKNOWN_ROUTE":
        route_colors.append(COLORS["orange"])
    else:
        route_colors.append(COLORS["blue"])

plt.figure(figsize=(9, 6.3))
plt.barh(
    route_plot["Route"],
    route_plot["Documents"],
    color=route_colors,
    height=0.66,
)

# Annotate only because counts are the point of this plot
annotate_horizontal_bars(plt.gca())

finish_plot(
    "Distribution of Document Routes",
    xlabel="Labeled documents",
    grid=False,
    legend=False,
    save_name="slide_06_document_routes.png",
)

In [ ]:
# ============================================================
# 27. Layout scatter — aspect ratio × OCR item count
# ============================================================
family_color = {
    "MYKAD": COLORS["blue"],
    "MRZ_DOCUMENT": COLORS["green"],
    "GENERIC_ID": COLORS["orange"],
}

plt.figure(figsize=(8.8, 5.8))

for family in ["MYKAD", "MRZ_DOCUMENT", "GENERIC_ID"]:
    sub = eda[eda["document_family"] == family]

    plt.scatter(
        sub["aspect_ratio_raw"],
        sub["ocr_item_count"],
        s=30,
        alpha=0.6,
        color=family_color[family],
        label=family.replace("_", " ").title(),
    )

plt.legend(frameon=False)

finish_plot(
    "Aspect Ratio vs OCR Item Count",
    xlabel="Image aspect ratio (width / height)",
    ylabel="Detected OCR item count",
    grid=True,
    legend=True,
)

In [ ]:
# ============================================================
# 28. Representative route montage
# ============================================================
route_order = (
    route_counts
    .sort_values(
        "Documents",
        ascending=False,
    )["Route"]
    .tolist()
)

route_gallery_items = []

for route in route_order[:9]:
    sub = eda[
        eda["route_final"] == route
    ].copy()

    # Representative = closest to median OCR-item count
    median_count = (
        sub["ocr_item_count"].median()
    )

    idx = (
        sub["ocr_item_count"]
        .sub(median_count)
        .abs()
        .idxmin()
    )

    row = sub.loc[idx]

    route_gallery_items.append({
        "filename": row["filename"],
        "label": route.replace(
            "_",
            " ",
        ).title(),
        "detail": (
            f'{row["filename"]} | '
            f'{int(row["ocr_item_count"])} OCR boxes'
        ),
    })

route_montage = make_montage(
    route_gallery_items,
    columns=3,
)

route_montage_path = (
    SLIDE_DIR
    / "slide_07_route_examples_montage.jpg"
)

route_montage.save(
    route_montage_path,
    quality=92,
)

display(route_montage)
print("Saved:", route_montage_path)

### Key finding → pipeline implication

**KEY FINDING**  
The dataset contains multiple stable structural families and a substantial unknown-layout tail.

**MODELING IMPLICATION**  
Field location, order, metadata cues, and parsing rules change by document family. A single global parser must compromise across incompatible layouts.

**PIPELINE RESPONSE**  
The final architecture first routes the document, then invokes a **specialist extractor** appropriate to that family.

# 8. Field Availability × Document Route

Document family changes not only *where* a field appears, but also *whether the field is expected to exist*.

Address is the clearest example.

In [ ]:
# ============================================================
# 29. Address presence by detailed route
# ============================================================
route_address = (
    eda.groupby("route_final")
    .agg(
        Documents=(
            "filename",
            "size",
        ),
        Address_present=(
            "address_present_gt",
            "sum",
        ),
        Address_rate=(
            "address_present_gt",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "Address_rate",
            "Documents",
        ],
        ascending=[
            False,
            False,
        ],
    )
)

display(
    route_address.style
    .format({
        "Address_rate": "{:.1%}"
    })
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 30. Address presence rate by route
# ============================================================
plot_addr = route_address.sort_values(
    "Address_rate",
    ascending=True,
)

colors = [
    COLORS["blue"]
    if r > 0.5
    else COLORS["orange"]
    if r > 0
    else COLORS["gray"]
    for r in plot_addr["Address_rate"]
]

plt.figure(figsize=(9, 6.3))
plt.barh(
    plot_addr["route_final"],
    100 * plot_addr["Address_rate"],
    color=colors,
    height=0.66,
)

finish_plot(
    "Address Availability by Document Route",
    xlabel="Address-present rate (%)",
    grid=False,
    legend=False,
)

### Pipeline implication

This is direct evidence that **routing is not cosmetic**.

For example, a route can imply:
- Address should be extracted,
- Address should usually be blank,
- MRZ parsing is available,
- field-number semantics may exist,
- a generic parser must be used as fallback.

That makes specialist extraction more defensible than one universal rule.

# 9. Name Complexity

Name is the hardest field to generalize across layouts.

Raw GT allows us to analyze:
- character length,
- token count,
- punctuation / hyphen structure.

This does **not** mean “pick the longest text as Name”.

Length is only one discriminative signal that can be combined with:
- geometry,
- OCR confidence,
- text shape,
- proximity to identifiers / anchors,
- route-specific layout cues.

That matches the final MyKad name-span ranker design.

In [ ]:
# ============================================================
# 31. Raw Name complexity features
# ============================================================
name_eda = (
    gt[
        [
            "filename",
            "name",
        ]
    ]
    .merge(
        eda[
            [
                "filename",
                "route_final",
            ]
        ],
        on="filename",
        how="left",
    )
)

name_eda["name_norm"] = (
    name_eda["name"]
    .map(norm_text)
)

name_eda["char_length"] = (
    name_eda["name_norm"]
    .str.len()
)

name_eda["token_count"] = (
    name_eda["name_norm"]
    .str.split()
    .str.len()
)

name_eda["has_hyphen"] = (
    name_eda["name_norm"]
    .str.contains(
        "-",
        regex=False,
    )
)

name_summary = (
    name_eda[
        [
            "char_length",
            "token_count",
        ]
    ]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.5,
            0.75,
            0.95,
        ]
    )
)

display(
    name_summary.round(2)
)

In [ ]:
# ============================================================
# 32. Name character-length distribution
# ============================================================
plt.figure(figsize=(8.8, 5.2))

counts, bins, patches = plt.hist(
    name_eda["char_length"],
    bins=np.arange(
        name_eda["char_length"].min(),
        name_eda["char_length"].max() + 3,
        3,
    ),
    edgecolor="white",
)

for i, p in enumerate(patches):
    shade = BLUE_SHADES[min(
        int(i / max(len(patches), 1) * len(BLUE_SHADES)),
        len(BLUE_SHADES) - 1
    )]
    p.set_facecolor(shade)

plt.axvline(
    name_eda["char_length"].median(),
    color=COLORS["orange"],
    linestyle="--",
    linewidth=2,
    label=f'Median = {name_eda["char_length"].median():.0f}',
)

plt.legend(frameon=False)

finish_plot(
    "Distribution of Name Character Length",
    xlabel="Number of characters",
    ylabel="Frequency",
    grid=True,
    legend=True,
)

In [ ]:
# ============================================================
# 33. Name token-count distribution
# ============================================================
token_counts = (
    name_eda["token_count"]
    .value_counts()
    .sort_index()
)

colors = [
    BLUE_SHADES[min(i, len(BLUE_SHADES)-1)]
    for i in range(len(token_counts))
]

plt.figure(figsize=(8.4, 5.0))
plt.bar(
    token_counts.index.astype(str),
    token_counts.values,
    color=colors,
    width=0.65,
)

finish_plot(
    "Distribution of Name Token Count",
    xlabel="Number of tokens",
    ylabel="Frequency",
    grid=True,
    legend=False,
)

# 10. Within-Identity Capture Variability

Repeated images only become useful for peer consensus if captures of the same identity are **not exact duplicates in quality**.

Kita mengukur spread within repeated groups untuk:
- brightness,
- blur score,
- OCR confidence.

In [ ]:
# ============================================================
# 34. Within-identity quality spread
# ============================================================
repeated_ids = (
    captures[
        captures > 1
    ].index
)

rep = eda[
    eda["identity_group"]
    .isin(repeated_ids)
].copy()

within_spread = (
    rep.groupby(
        "identity_group"
    )
    .agg(
        Captures=(
            "filename",
            "size",
        ),
        Brightness_range=(
            "brightness_raw",
            lambda s: s.max() - s.min(),
        ),
        Blur_ratio=(
            "blur_score_raw",
            lambda s: (
                s.max()
                / max(s.min(), 1e-6)
            ),
        ),
        OCR_conf_range=(
            "ocr_confidence",
            lambda s: s.max() - s.min(),
        ),
        Name_exact_mean=(
            "name_exact",
            "mean",
        ),
        DOB_exact_mean=(
            "dob_exact",
            "mean",
        ),
    )
    .reset_index()
)

display(
    within_spread[
        [
            "Captures",
            "Brightness_range",
            "Blur_ratio",
            "OCR_conf_range",
            "Name_exact_mean",
            "DOB_exact_mean",
        ]
    ]
    .describe()
    .round(3)
)

In [ ]:
# ============================================================
# 35. Within-identity brightness spread
# ============================================================
plt.figure(figsize=(8.8, 5.2))

counts, bins, patches = plt.hist(
    within_spread["Brightness_range"],
    bins=18,
    edgecolor="white",
)

for i, p in enumerate(patches):
    shade = BLUE_SHADES[min(
        int(i / max(len(patches), 1) * len(BLUE_SHADES)),
        len(BLUE_SHADES) - 1
    )]
    p.set_facecolor(shade)

median_range = within_spread["Brightness_range"].median()

plt.axvline(
    median_range,
    color=COLORS["orange"],
    linestyle="--",
    linewidth=2,
    label=f"Median = {median_range:.1f}",
)

plt.legend(frameon=False)

finish_plot(
    "Within-Identity Brightness Variation",
    xlabel="Brightness range within identity group",
    ylabel="Frequency",
    grid=True,
    legend=True,
)

In [ ]:
# ============================================================
# 36. Hero montage — one repeated identity with strong quality spread
# ============================================================
candidate_group = (
    within_spread
    .sort_values(
        [
            "Brightness_range",
            "OCR_conf_range",
        ],
        ascending=False,
    )
    .iloc[0][
        "identity_group"
    ]
)

group_rows = (
    rep[
        rep["identity_group"]
        == candidate_group
    ]
    .sort_values(
        "brightness_raw"
    )
)

# Select up to 6 captures spread across ordering
positions = np.linspace(
    0,
    len(group_rows) - 1,
    min(6, len(group_rows)),
).round().astype(int)

selected_group_rows = (
    group_rows.iloc[
        np.unique(positions)
    ]
)

group_gallery_items = []

for _, row in (
    selected_group_rows.iterrows()
):
    group_gallery_items.append({
        "filename": row["filename"],
        "label": row["filename"],
        "detail": (
            f'Brightness {row["brightness_raw"]:.0f} | '
            f'OCR conf {row["ocr_confidence"]:.2f}'
        ),
    })

group_montage = make_montage(
    group_gallery_items,
    columns=3,
)

group_montage_path = (
    SLIDE_DIR
    / "slide_08_same_identity_variation.jpg"
)

group_montage.save(
    group_montage_path,
    quality=92,
)

display(group_montage)

print("Identity group:", candidate_group)
print("Saved:", group_montage_path)

### Key finding → pipeline implication

**KEY FINDING**  
Repeated captures of one label-derived identity can differ substantially in brightness, sharpness, and OCR confidence.

**MODELING IMPLICATION**  
Single-image inference may fail on one capture even when another capture of the same identity contains stronger evidence.

**PIPELINE RESPONSE**  
Identity linking + conservative peer consensus can exploit cross-capture redundancy.

# 11. Identity-Linking Evidence from OCR Text

Final linker uses a richer representation:

```text
OCR text + predicted fields
→ TF-IDF character n-gram 2–5
→ SVD
+ quality / OCR metadata
+ document-family one-hot
→ Siamese network
→ 64D embedding
→ cosine similarity
```

EDA tidak akan retrain Siamese.

Sebagai diagnostic evidence, kita cek apakah **character n-gram OCR representation alone** already contains identity signal:

- same identity-group pair,
- different identity-group pair.

Jika same-identity OCR text tends to be more similar, maka embedding-based identity linking memiliki basis data yang masuk akal.

In [ ]:
# ============================================================
# 37. Character TF-IDF representation
# ============================================================
ocr_texts = (
    eda["ocr_text"]
    .fillna("")
    .astype(str)
)

tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    min_df=2,
    max_features=30000,
    lowercase=True,
    sublinear_tf=True,
)

X = tfidf.fit_transform(
    ocr_texts
)

print(
    "TF-IDF matrix:",
    X.shape,
)
print(
    "Vocabulary size:",
    len(tfidf.vocabulary_),
)

In [ ]:
# ============================================================
# 38. Sample same-identity vs different-identity pairs
# ============================================================
group_to_indices = {
    group: idx.to_numpy()
    for group, idx in (
        eda.reset_index()
        .groupby(
            "identity_group"
        )["index"]
    )
}

same_pairs = []

for group, idx in (
    group_to_indices.items()
):
    if len(idx) < 2:
        continue

    pairs = list(
        combinations(
            idx,
            2,
        )
    )

    # Cap huge groups to keep EDA efficient
    if len(pairs) > 1000:
        chosen = rng.choice(
            len(pairs),
            size=1000,
            replace=False,
        )
        pairs = [
            pairs[i]
            for i in chosen
        ]

    same_pairs.extend(
        pairs
    )

same_pairs = np.asarray(
    same_pairs,
    dtype=int,
)

n_target = len(
    same_pairs
)

different_pairs = []

all_idx = np.arange(
    len(eda)
)

while len(different_pairs) < n_target:
    a = int(
        rng.choice(all_idx)
    )
    b = int(
        rng.choice(all_idx)
    )

    if a == b:
        continue

    if (
        eda.iloc[a]["identity_group"]
        == eda.iloc[b]["identity_group"]
    ):
        continue

    different_pairs.append(
        (a, b)
    )

different_pairs = np.asarray(
    different_pairs,
    dtype=int,
)

print(
    "Same-identity pairs     :",
    len(same_pairs),
)
print(
    "Different-identity pairs:",
    len(different_pairs),
)

In [ ]:
# ============================================================
# 39. Pairwise cosine similarity
# ============================================================
def rowwise_sparse_cosine(
    matrix,
    pairs,
    batch_size=4096,
):
    sims = []

    for start in range(
        0,
        len(pairs),
        batch_size,
    ):
        batch = pairs[
            start:start+batch_size
        ]

        left = matrix[
            batch[:, 0]
        ]
        right = matrix[
            batch[:, 1]
        ]

        # TF-IDF is L2 normalized by default
        vals = np.asarray(
            left.multiply(
                right
            ).sum(axis=1)
        ).ravel()

        sims.extend(
            vals.tolist()
        )

    return np.asarray(sims)


same_sim = rowwise_sparse_cosine(
    X,
    same_pairs,
)

diff_sim = rowwise_sparse_cosine(
    X,
    different_pairs,
)

labels = np.r_[
    np.ones(len(same_sim)),
    np.zeros(len(diff_sim)),
]

scores = np.r_[
    same_sim,
    diff_sim,
]

pair_auc = roc_auc_score(
    labels,
    scores,
)

similarity_summary = pd.DataFrame({
    "Pair type": [
        "Same identity group",
        "Different identity group",
    ],
    "Pairs": [
        len(same_sim),
        len(diff_sim),
    ],
    "Median cosine": [
        np.median(same_sim),
        np.median(diff_sim),
    ],
    "Mean cosine": [
        same_sim.mean(),
        diff_sim.mean(),
    ],
})

display(
    similarity_summary.style
    .format({
        "Median cosine": "{:.3f}",
        "Mean cosine": "{:.3f}",
    })
    .hide(axis="index")
)

print(
    f"Diagnostic char-TF-IDF pair AUC = {pair_auc:.3f}"
)

In [ ]:
# ============================================================
# 40. Same vs different OCR similarity
# ============================================================
bins = np.linspace(0, 1, 32)

plt.figure(figsize=(8.8, 5.2))

plt.hist(
    diff_sim,
    bins=bins,
    density=True,
    alpha=0.45,
    color=COLORS["gray"],
    label="Different identity group",
)

plt.hist(
    same_sim,
    bins=bins,
    density=True,
    alpha=0.62,
    color=COLORS["blue"],
    label="Same identity group",
)

plt.legend(frameon=False)

finish_plot(
    "OCR Similarity: Same vs Different Identity Groups",
    xlabel="Cosine similarity",
    ylabel="Density",
    grid=False,
    legend=True,
    save_name="slide_09_identity_similarity.png",
)

### Key finding → pipeline implication

**KEY FINDING**  
Same-group OCR text is measurably more similar than unrelated-group OCR text even before Siamese learning.

**MODELING IMPLICATION**  
OCR character patterns can serve as a meaningful identity-link signal, but raw text similarity alone is imperfect because capture quality and OCR errors overlap the distributions.

**PIPELINE RESPONSE**  
The final linker enriches char TF-IDF/SVD with OCR quality, box count, field presence, and document-family features, then learns a **Siamese 64D embedding** with a conservative clustering threshold.

# 12. Why Peer Consensus Can Help

Peer consensus only makes sense if repeated identity groups sometimes contain:

- a correct capture,
- and an incorrect capture.

If every capture were always correct or always wrong, peer voting would add little.

Kita cek mixed correctness pada repeated groups untuk:
- Name,
- DOB.

In [ ]:
# ============================================================
# 41. Mixed correctness within repeated identity groups
# ============================================================
peer_group_stats = (
    rep.groupby(
        "identity_group"
    )
    .agg(
        Captures=(
            "filename",
            "size",
        ),
        Name_correct=(
            "name_exact",
            "sum",
        ),
        DOB_correct=(
            "dob_exact",
            "sum",
        ),
    )
    .reset_index()
)

def correctness_state(
    correct,
    total,
):
    if correct == 0:
        return "All wrong"
    if correct == total:
        return "All correct"
    return "Mixed"


peer_group_stats[
    "Name_state"
] = [
    correctness_state(
        c,
        n,
    )
    for c, n in zip(
        peer_group_stats[
            "Name_correct"
        ],
        peer_group_stats[
            "Captures"
        ],
    )
]

peer_group_stats[
    "DOB_state"
] = [
    correctness_state(
        c,
        n,
    )
    for c, n in zip(
        peer_group_stats[
            "DOB_correct"
        ],
        peer_group_stats[
            "Captures"
        ],
    )
]

state_order = [
    "All wrong",
    "Mixed",
    "All correct",
]

state_table = pd.DataFrame({
    "State": state_order,
    "Name groups": [
        (
            peer_group_stats[
                "Name_state"
            ] == s
        ).sum()
        for s in state_order
    ],
    "DOB groups": [
        (
            peer_group_stats[
                "DOB_state"
            ] == s
        ).sum()
        for s in state_order
    ],
})

display(
    state_table.style
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 42. Peer-consensus opportunity
# ============================================================
x = np.arange(len(state_table))
width = 0.36

plt.figure(figsize=(8.5, 5.0))

plt.bar(
    x - width/2,
    state_table["Name groups"],
    width,
    color=COLORS["blue"],
    label="Name",
)

plt.bar(
    x + width/2,
    state_table["DOB groups"],
    width,
    color=COLORS["orange"],
    label="DOB",
)

plt.xticks(x, state_table["State"])
plt.legend(frameon=False)

finish_plot(
    "Prediction Consistency Within Repeated Identity Groups",
    ylabel="Repeated identity groups",
    grid=True,
    legend=True,
    save_name="slide_10_peer_consensus_opportunity.png",
)

In [ ]:
# ============================================================
# 43. Example repeated group with mixed DOB/Name correctness
# ============================================================
mixed_candidates = (
    peer_group_stats[
        (
            peer_group_stats[
                "Name_state"
            ].eq("Mixed")
        )
        | (
            peer_group_stats[
                "DOB_state"
            ].eq("Mixed")
        )
    ]
    .sort_values(
        "Captures",
        ascending=False,
    )
)

if len(mixed_candidates):
    example_group = (
        mixed_candidates.iloc[0][
            "identity_group"
        ]
    )

    example_peer_rows = (
        rep[
            rep["identity_group"]
            == example_group
        ][
            [
                "filename",
                "name_gt",
                "name_pred",
                "name_exact",
                "birth_date_gt",
                "birth_date_pred",
                "dob_exact",
                "ocr_confidence",
                "quality_group",
            ]
        ]
        .sort_values(
            "filename"
        )
        .head(20)
    )

    print(
        "Example mixed group:",
        example_group,
    )

    display(
        example_peer_rows
        .style
        .hide(axis="index")
    )

else:
    print(
        "No mixed correctness group found."
    )

### Key finding → pipeline implication

**KEY FINDING**  
Within some repeated identity groups, certain scans are correct while peers are wrong.

**MODELING IMPLICATION**  
Information is distributed unevenly across captures.

**PIPELINE RESPONSE**  
The final system uses **leave-one-out peer consensus**:
- target image never votes for itself,
- consensus is gated,
- only validated routes / fields are changed,
- replacement happens only when evidence is sufficiently consistent.

# 13. Pipeline-Assisted Performance by Route

EDA seharusnya tidak hanya berkata “layout berbeda”.

Jika route-specific specialist logic memang diperlukan, performance profile juga seharusnya berbeda antar route.

Kita gunakan frozen V28 evaluation metrics hanya sebagai **diagnostic validation of heterogeneity**, bukan untuk memilih ulang model.

In [ ]:
# ============================================================
# 44. Route-level field performance
# ============================================================
route_perf = (
    eda.groupby(
        "route_final"
    )
    .agg(
        Documents=(
            "filename",
            "size",
        ),
        Name_EM=(
            "name_exact",
            "mean",
        ),
        Name_similarity=(
            "name_similarity",
            "mean",
        ),
        DOB_EM=(
            "dob_exact",
            "mean",
        ),
        DOB_similarity=(
            "dob_similarity",
            "mean",
        ),
    )
    .reset_index()
    .sort_values(
        "Documents",
        ascending=False,
    )
)

display(
    route_perf.style
    .format({
        "Name_EM": "{:.1%}",
        "Name_similarity": "{:.1%}",
        "DOB_EM": "{:.1%}",
        "DOB_similarity": "{:.1%}",
    })
    .hide(axis="index")
)

In [ ]:
# ============================================================
# 45. Route-level Name performance
# ============================================================
plot_perf = route_perf.sort_values(
    "Name_similarity",
    ascending=True,
)

colors = [
    COLORS["orange"] if r == "UNKNOWN_ROUTE"
    else COLORS["blue"]
    for r in plot_perf["route_final"]
]

plt.figure(figsize=(9, 6.3))
plt.barh(
    plot_perf["route_final"],
    100 * plot_perf["Name_similarity"],
    color=colors,
    height=0.66,
)

finish_plot(
    "Name Similarity by Document Route",
    xlabel="Mean normalized edit similarity (%)",
    grid=False,
    legend=False,
)

### Diagnostic interpretation

Strong differences between route performance reinforce the architectural choice:

```text
detect document family
        ↓
specialist extractor if a stable route exists
        ↓
generic fallback only when necessary
```

The `UNKNOWN_ROUTE` tail is not evidence that routing failed; it identifies the region where stable structure was not discovered.

# 14. EDA → Final Pipeline Evidence Map

This is the final technical bridge from data understanding to modeling.

The goal is to make every component answer a **specific observed property of the dataset**.

In [ ]:
# ============================================================
# 46. Evidence-to-architecture map
# ============================================================
evidence_map = pd.DataFrame([
    {
        "EDA evidence": (
            "Brightness, blur, glare, and illumination "
            "span multiple regimes."
        ),
        "Technical risk": (
            "One universal preprocessing transform can "
            "over-process clean captures or under-process degraded ones."
        ),
        "Pipeline response": (
            "Quality-aware Raw / Gray / Gray+CLAHE selection"
        ),
    },
    {
        "EDA evidence": (
            "Documents separate into MRZ, MyKad, driving licence, "
            "country-specific IDs, and an unknown-layout tail."
        ),
        "Technical risk": (
            "Field location/order/labels are incompatible across layouts."
        ),
        "Pipeline response": (
            "Document-family routing + specialist parsers"
        ),
    },
    {
        "EDA evidence": (
            "Name character/token lengths vary widely."
        ),
        "Technical risk": (
            "Hard text-length rules cannot reliably identify Name."
        ),
        "Pipeline response": (
            "Candidate ranking using length + geometry + confidence + text shape"
        ),
    },
    {
        "EDA evidence": (
            "632 labeled images form 152 pipeline identity groups; "
            "506 images belong to repeated groups."
        ),
        "Technical risk": (
            "Random image validation leaks identity; single-image inference wastes redundancy."
        ),
        "Pipeline response": (
            "Identity-safe validation + identity linker"
        ),
    },
    {
        "EDA evidence": (
            "Same-group captures vary in quality and OCR confidence."
        ),
        "Technical risk": (
            "A bad scan can fail even when another scan of the same identity is informative."
        ),
        "Pipeline response": (
            "Siamese identity embedding + conservative complete-link clustering"
        ),
    },
    {
        "EDA evidence": (
            "Repeated groups contain mixed correct/incorrect predictions."
        ),
        "Technical risk": (
            "Useful information is distributed across captures."
        ),
        "Pipeline response": (
            "Leave-one-out peer medoid / consensus with validation gates"
        ),
    },
])

display(
    evidence_map.style
    .hide(axis="index")
    .set_properties(
        **{
            "white-space": "normal",
            "text-align": "left",
        }
    )
)

# 15. ⭐ Slide-Ready EDA

Notebook lengkap di atas adalah technical evidence.

Untuk presentasi, **jangan pindahkan semua grafik**. Gunakan empat hero findings yang paling langsung menjelaskan pipeline.

### Recommended storyline

#### Hero 1 — Capture quality is heterogeneous
**Use:** `slide_03_quality_groups.png` + optional quality montage  
**Takeaway:** “Capture degradation is not uniform, so preprocessing should be conditional rather than universal.”  
**Pipeline:** Raw / Gray / CLAHE quality-aware selection.

#### Hero 2 — Document structure is heterogeneous
**Use:** `slide_06_document_routes.png` + `slide_07_route_examples_montage.jpg`  
**Takeaway:** “The dataset contains multiple structural document families, making one universal parser unrealistic.”  
**Pipeline:** document-family routing + specialist extractors.

#### Hero 3 — Repeated captures are a dominant dataset structure
**Use:** `slide_02_identity_repetition.png`  
**Takeaway:** “632 labeled images correspond to 152 label-derived identity groups, and repeated groups contain most labeled images.”  
**Pipeline:** identity-safe validation + identity linking.

#### Hero 4 — Cross-capture information is usable
**Use:** `slide_09_identity_similarity.png` and/or `slide_10_peer_consensus_opportunity.png`  
**Takeaway:** “Same-identity OCR patterns are more similar, while repeated groups contain mixed-quality predictions — enabling conservative peer consensus.”  
**Pipeline:** TF-IDF/SVD + Siamese linker → complete-link clustering → leave-one-out consensus.

In [ ]:
# ============================================================
# 47. Slide-ready summary table
# ============================================================
slide_ready = pd.DataFrame([
    {
        "Slide finding": "1. Capture-quality heterogeneity",
        "Core evidence": (
            f'{(eda["quality_group"] != "clean").mean():.1%} '
            "of labeled documents are assigned a non-clean "
            "quality regime by the frozen pipeline."
        ),
        "Recommended visual": (
            "slide_03_quality_groups.png + "
            "slide_05_quality_extremes_montage.jpg"
        ),
        "Pipeline justification": (
            "Quality-aware Raw / Gray / CLAHE preprocessing"
        ),
    },
    {
        "Slide finding": "2. Structural document heterogeneity",
        "Core evidence": (
            f'{eda["route_final"].nunique()} diagnostic routes '
            "across MyKad, MRZ, driving licences, "
            "country-specific IDs, and unknown layouts."
        ),
        "Recommended visual": (
            "slide_06_document_routes.png + "
            "slide_07_route_examples_montage.jpg"
        ),
        "Pipeline justification": (
            "Document routing + specialist extractors"
        ),
    },
    {
        "Slide finding": "3. Identity repetition",
        "Core evidence": (
            f'632 labeled images → {n_pipeline_groups} '
            f'pipeline identity groups; '
            f'{repeated_rows} images belong to repeated groups.'
        ),
        "Recommended visual": (
            "slide_02_identity_repetition.png"
        ),
        "Pipeline justification": (
            "Identity-safe validation + identity linker"
        ),
    },
    {
        "Slide finding": "4. Cross-capture evidence",
        "Core evidence": (
            f'Char-TF-IDF identity-pair AUC = {pair_auc:.3f}; '
            f'{(peer_group_stats["Name_state"]=="Mixed").sum()} '
            "repeated groups have mixed Name correctness and "
            f'{(peer_group_stats["DOB_state"]=="Mixed").sum()} '
            "have mixed DOB correctness."
        ),
        "Recommended visual": (
            "slide_09_identity_similarity.png / "
            "slide_10_peer_consensus_opportunity.png"
        ),
        "Pipeline justification": (
            "Siamese identity linking + leave-one-out peer consensus"
        ),
    },
])

display(
    slide_ready.style
    .hide(axis="index")
    .set_properties(
        **{
            "white-space": "normal",
            "text-align": "left",
        }
    )
)

## One-slide condensed version

If EDA only gets **one presentation slide**, use this narrative:

> **The dataset is heterogeneous at three levels.** Capture quality spans clean, dark, blurred, glare, and uneven-light regimes; document structure spans multiple identity-document families; and 632 labeled scans map to only 152 label-derived identity groups with substantial repeated captures. Therefore, our final system adapts preprocessing to image quality, routes documents to specialist field extractors, and exploits repeated scans through conservative identity linking and leave-one-out peer consensus.

Suggested visual layout:

```text
┌──────────────────┬──────────────────┬──────────────────┐
│ Quality regimes  │ Document routes  │ Repeated scans   │
│      chart       │      chart       │      chart       │
└──────────────────┴──────────────────┴──────────────────┘
         ↓                  ↓                  ↓
 Adaptive preprocess   Specialist routes   Identity consensus
```

# 16. Export EDA Artifacts

Output:
- complete EDA summary tables,
- raw quality features,
- identity audit,
- route summaries,
- slide-ready evidence table,
- all saved presentation PNG/JPG assets.

In [ ]:
# ============================================================
# 48. Export EDA artifacts
# ============================================================
EXPORT_DIR = WORK_DIR / "eda_artifacts"
EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

raw_quality.to_csv(
    EXPORT_DIR / "raw_image_quality_eda.csv",
    index=False,
)

identity_count_table.to_csv(
    EXPORT_DIR / "identity_count_audit.csv",
    index=False,
)

identity_difference_explainer.to_csv(
    EXPORT_DIR / "identity_count_difference_explainer.csv",
    index=False,
)

capture_dist.to_csv(
    EXPORT_DIR / "captures_per_identity_group.csv",
    index=False,
)

route_counts.to_csv(
    EXPORT_DIR / "document_route_counts.csv",
    index=False,
)

route_address.to_csv(
    EXPORT_DIR / "address_presence_by_route.csv",
    index=False,
)

within_spread.to_csv(
    EXPORT_DIR / "within_identity_quality_spread.csv",
    index=False,
)

similarity_summary.to_csv(
    EXPORT_DIR / "identity_similarity_summary.csv",
    index=False,
)

peer_group_stats.to_csv(
    EXPORT_DIR / "peer_consensus_group_diagnostics.csv",
    index=False,
)

route_perf.to_csv(
    EXPORT_DIR / "route_performance_diagnostic.csv",
    index=False,
)

evidence_map.to_csv(
    EXPORT_DIR / "eda_to_pipeline_evidence_map.csv",
    index=False,
)

slide_ready.to_csv(
    EXPORT_DIR / "slide_ready_eda_summary.csv",
    index=False,
)

# Copy slide assets into export bundle
slide_export = EXPORT_DIR / "slide_ready_assets"
slide_export.mkdir(
    exist_ok=True,
)

for asset in SLIDE_DIR.iterdir():
    if asset.is_file():
        shutil.copy2(
            asset,
            slide_export / asset.name,
        )

zip_path = shutil.make_archive(
    str(
        WORK_DIR.parent
        / "Ketolak_Pacil_Pipeline_Driven_EDA_Artifacts"
    ),
    "zip",
    root_dir=EXPORT_DIR,
)

print("=" * 78)
print("EDA EXPORT COMPLETE")
print("=" * 78)
print("Artifact folder :", EXPORT_DIR)
print("Slide assets    :", slide_export)
print("ZIP             :", zip_path)
print("✅ Ready for technical report + presentation.")

# Final EDA Conclusion

The final pipeline is supported by three independent dataset properties:

1. **Capture heterogeneity**  
   → quality-aware preprocessing.

2. **Document-layout and field heterogeneity**  
   → routing + specialist extractors.

3. **Repeated identity structure with unequal capture quality**  
   → identity-safe validation + Siamese identity linking + leave-one-out consensus.

### Identity terminology to keep consistent

Use:

> **“152 label-derived pipeline identity groups across 632 labeled images.”**

Do **not** casually say “152 verified unique people”.

Also keep the audit distinction:

- **132 unique normalized Name strings**
- **152 unique normalized (Name, DOB) pairs**
- **152 identity groups in the final teammate pipeline**

This wording is both pipeline-consistent and technically defensible.